# Analysis Brainstorm

## Ideas of how to quantify bus "performance"

- Stop idle times
- Average bus speed
- Time to complete one full loop

## Compare SPRING 2026 (last semester) with FALL 2026 (this semester)

## Compare Housing Express and South Campus Express performance

## Compare route performance over the course of the day

In [ ]:
import glob
import os
import re

import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "data"
TIMEZONE = "America/Denver"
MAX_GAP_S = 60                  # pings further apart than this aren't treated as consecutive
LOOP_ANCHOR_STOP = "Lundstrum"  # served exactly once per loop on BOTH routes

ROUTE_NAMES = {"housing_express": "Housing Express", "south_campus": "South Campus Express"}

In [ ]:
def load_semester(semester):
    frames = []
    for path in sorted(glob.glob(f"{DATA_DIR}/{semester}/*.csv")):
        m = re.match(r"(housing_express|south_campus)_(\d+-\d+-\d+)\.csv", os.path.basename(path))
        if not m:
            continue
        df = pd.read_csv(path)
        if df.empty:
            print(f"Skipping empty file: {path}")
            continue
        df["route"] = ROUTE_NAMES[m.group(1)]
        df["date"] = pd.to_datetime(m.group(2), format="%m-%d-%y").date()
        frames.append(df)

    df = pd.concat(frames, ignore_index=True)
    df["time"] = pd.to_datetime(df["timestamp"], unit="s", utc=True).dt.tz_convert(TIMEZONE)
    df = df.sort_values(["route", "date", "vehicle_id", "timestamp"]).reset_index(drop=True)

    # A "run" is an unbroken stretch of recording for one bus. Visits and loops never span two runs,
    # so recording gaps (e.g. 10/2 around 10:20-10:50) can't create fake long idle times or loops.
    grp = df.groupby(["route", "date", "vehicle_id"])
    df["run_id"] = (grp.ngroup().diff().ne(0) | (grp["timestamp"].diff() > MAX_GAP_S)).cumsum()
    return df

df = load_semester("fall26")
df.head()

In [ ]:
df["visit_id"] = ((df["stop"] != df["stop"].shift()) | (df["run_id"] != df["run_id"].shift())).cumsum()

visits = (
    df[df["stop"] != "in_transit"]
      .groupby("visit_id")
      .agg(route=("route", "first"), run_id=("run_id", "first"), stop=("stop", "first"),
           arrive=("time", "first"), depart=("time", "last"), arrive_ts=("timestamp", "first"))
      .reset_index(drop=True)
)
visits["idle_s"] = (visits["depart"] - visits["arrive"]).dt.total_seconds()

print(f"{len(visits)} stop visits")
visits.head()

In [ ]:
idle_by_stop = (
    visits.groupby(["route", "stop"])["idle_s"]
          .agg(visits="size", median="median", mean="mean", p90=lambda s: s.quantile(0.9), max="max")
          .round(1)
          .sort_values(["route", "median"], ascending=[True, False])
)
idle_by_stop

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (route, g) in zip(axes, visits.groupby("route")):
    order = g.groupby("stop")["idle_s"].median().sort_values().index
    ax.boxplot([g.loc[g["stop"] == s, "idle_s"] for s in order], orientation="horizontal", showfliers=False)
    ax.set_yticks(range(1, len(order) + 1), order)
    ax.set_xlabel("Idle time (s)")
    ax.set_title(route)
plt.suptitle("Idle time by stop (outliers hidden)")
plt.tight_layout()
plt.show()

In [ ]:
loops = visits[visits["stop"] == LOOP_ANCHOR_STOP].sort_values("arrive_ts").copy()
loops["loop_min"] = loops.groupby("run_id")["arrive_ts"].diff() / 60
loops = loops.dropna(subset=["loop_min"])
loops["hour"] = loops["arrive"].dt.hour

# Flag loops way longer than normal (missed Lundstrum ping or a driver break)
loops["suspect"] = loops["loop_min"] > 1.5 * loops.groupby("route")["loop_min"].transform("median")
print(f"Suspect loops removed: {loops['suspect'].sum()}")

clean_loops = loops[~loops["suspect"]]
clean_loops.groupby("route")["loop_min"].describe().round(2)

In [ ]:
hourly_loops = (
    clean_loops.groupby(["route", "hour"])["loop_min"]
               .agg(loops="size", median="median",
                    q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75))
               .round(2)
)
hourly_loops

In [ ]:
def hour_label(h):
    suffix = "AM" if h < 12 else "PM"
    return f"{(h - 1) % 12 + 1} {suffix}"

hours = range(7, 18)  # 7 AM to 5 PM

fig, ax = plt.subplots(figsize=(9, 5))
for route, g in hourly_loops.reset_index().groupby("route"):
    line, = ax.plot(g["hour"], g["median"], marker="o", label=route)
    ax.fill_between(g["hour"], g["q25"], g["q75"], color=line.get_color(), alpha=0.15)
ax.set_xlabel("Hour of day (loop completed)")
ax.set_ylabel("Loop time (min)")
ax.set_title("Loop time over the day (median, shaded = middle 50%)")
ax.set_xticks(hours, [hour_label(h) for h in hours])
ax.grid(alpha=0.3)
ax.legend()
plt.show()